In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# DATABASE + IB CONNECTION
# ══════════════════════════════════════════════════════════════════════════════

client             = MongoClient('mongodb://localhost:27017/')
db                 = client['dipBuy_db']
trades_collection  = db['trades_cardwell_rsi']
exclude_collection = db['excluded_tickers_cardwell_rsi']

util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))

tickers = [
    'WMT', 'MU', 'SAVA', 'SLDB', 'SLV', 'NEM', 'CTXR', 'ONON', 'IONQ', 'MARA',
    'MRVL', 'T', 'CCL', 'XYZ', 'PG', 'ONDS', 'NFLX', 'V', 'ADBE', 'MS', 'CXW',
    'BA', 'BABA', 'DAL', 'JPM', 'C', 'AMZN', 'UAL', 'BRK.B', 'IBM', 'MSFT', 'KO',
    'MSTR', 'COIN', 'GLD', 'META', 'AAL', 'OSCR', 'QSI', 'SPY', 'NVO', 'DIS',
    'AAPL', 'BMNR', 'GRAB', 'RBLX', 'AFRM', 'NVDA', 'FUBO', 'PYPL', 'GOOG', 'BTC',
    'RGTI', 'GPRO', 'OKLO', 'AMD', 'XPEV', 'SHEL', 'TSM', 'SNOW', 'NET', 'SES',
    'TSLA', 'BP', 'UBER', 'INTC', 'MRNA', 'IREN', 'ORCL', 'HIMS', 'NBIS'
]
contracts = [Stock(t, 'SMART', 'USD') for t in tickers]
ib.qualifyContracts(*contracts)


# ══════════════════════════════════════════════════════════════════════════════
# STRATEGY PARAMETERS  — Cardwell RSI Range Shift
# ══════════════════════════════════════════════════════════════════════════════

RSI_PERIOD   = 14    # Standard RSI period

# ── Trend classification (daily RSI range over N bars) ──────────────────────
TREND_LOOKBACK = 50  # bars used to confirm RSI range regime

# ── Uptrend RSI range: RSI oscillates between 40–80 ─────────────────────────
UP_RSI_FLOOR   = 40  # RSI must stay above this to confirm uptrend
UP_RSI_CEIL    = 80  # RSI upper bound in uptrend

# ── Downtrend RSI range: RSI oscillates between 20–60 ───────────────────────
DN_RSI_FLOOR   = 20  # RSI lower bound in downtrend
DN_RSI_CEIL    = 60  # RSI must stay below this to confirm downtrend

# ── Entry zones ──────────────────────────────────────────────────────────────
LONG_ENTRY_LOW  = 40  # Buy when RSI pulls back into this zone …
LONG_ENTRY_HIGH = 50  # … uptrend support band

SHORT_ENTRY_LOW  = 50  # Sell when RSI rallies into this zone …
SHORT_ENTRY_HIGH = 60  # … downtrend resistance band

# ── Exit levels ──────────────────────────────────────────────────────────────
LONG_EXIT_RSI  = 70  # Close long when RSI reaches upper range
SHORT_EXIT_RSI = 30  # Cover short when RSI falls to lower range

# ── Misc ─────────────────────────────────────────────────────────────────────
ORDER_SIZE = 40      # Fixed shares per trade
MA_PERIOD  = 200     # Secondary trend confirmation (daily SMA)


# ══════════════════════════════════════════════════════════════════════════════
# ORDER HELPER  — always DAY to prevent TWS preset overriding TIF to GTC
# ══════════════════════════════════════════════════════════════════════════════

def day_market_order(action: str, qty: int) -> MarketOrder:
    """
    Return a MarketOrder with tif='DAY' explicitly set.
    Prevents TWS order presets from silently converting TIF to GTC
    (error 10349), which causes the order to be cancelled immediately.
    """
    order     = MarketOrder(action, qty)
    order.tif = 'DAY'
    return order


# ══════════════════════════════════════════════════════════════════════════════
# MONGO SANITIZER
# ══════════════════════════════════════════════════════════════════════════════

def sanitize_for_mongo(d):
    """Recursively convert numpy types to native Python types."""
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out[k] = sanitize_for_mongo(v)
        elif hasattr(v, 'item'):
            out[k] = v.item()
        else:
            out[k] = v
    return out


# ══════════════════════════════════════════════════════════════════════════════
# INDICATORS
# ══════════════════════════════════════════════════════════════════════════════

def calc_rsi(series: pd.Series, period: int = RSI_PERIOD) -> pd.Series:
    """Wilder-smoothed RSI."""
    d    = series.diff()
    gain = d.clip(lower=0).ewm(com=period - 1, min_periods=period).mean()
    loss = (-d.clip(upper=0)).ewm(com=period - 1, min_periods=period).mean()
    return 100 - 100 / (1 + gain / (loss + 1e-10))


def classify_trend(rsi_series: pd.Series, lookback: int = TREND_LOOKBACK) -> str:
    """
    Cardwell Range Shift regime detection.

    Examine the last `lookback` RSI values:
      • If the RSI min ≥ UP_RSI_FLOOR (40) and max ≤ UP_RSI_CEIL (80)
        → UPTREND  (RSI locked in the 40–80 bullish range)
      • If the RSI min ≥ DN_RSI_FLOOR (20) and max ≤ DN_RSI_CEIL (60)
        → DOWNTREND  (RSI locked in the 20–60 bearish range)
      • Otherwise → NEUTRAL (transitioning / no clear range)

    A relaxed version scores each condition proportionally and picks
    whichever regime has more bars satisfying its bounds.
    """
    window = rsi_series.iloc[-lookback:]
    if len(window) < lookback:
        return 'NEUTRAL'

    up_bars = ((window >= UP_RSI_FLOOR) & (window <= UP_RSI_CEIL)).sum()
    dn_bars = ((window >= DN_RSI_FLOOR) & (window <= DN_RSI_CEIL)).sum()

    threshold = lookback * 0.70   # 70 % of bars must fit the range

    if up_bars >= threshold and dn_bars < threshold:
        return 'UPTREND'
    if dn_bars >= threshold and up_bars < threshold:
        return 'DOWNTREND'
    if up_bars >= threshold and dn_bars >= threshold:
        # Overlapping — use most recent RSI to break tie
        last = float(rsi_series.iloc[-1])
        return 'UPTREND' if last > 50 else 'DOWNTREND'
    return 'NEUTRAL'


# ══════════════════════════════════════════════════════════════════════════════
# DATA FETCHING
# ══════════════════════════════════════════════════════════════════════════════

def get_daily_data(contract, ma_period: int = MA_PERIOD, rsi_period: int = RSI_PERIOD,
                   trend_lookback: int = TREND_LOOKBACK):
    """
    Fetch daily bars sufficient to compute:
      • 200-day SMA  (secondary trend confirmation)
      • RSI(14) range regime  (Cardwell range shift)

    Returns (sma200, daily_close, regime) or (None, None, None) on failure.
    """
    needed = max(ma_period, rsi_period + trend_lookback) + 20
    bars   = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr=f'{needed} D',
        barSizeSetting='1 day',
        whatToShow='TRADES',
        useRTH=True,
        formatDate=1
    )
    if not bars or len(bars) < rsi_period + trend_lookback:
        return None, None, None

    df         = util.df(bars)
    df.columns = [c.lower() for c in df.columns]
    df['RSI']  = calc_rsi(df['close'], period=rsi_period)
    df.dropna(subset=['RSI'], inplace=True)

    sma200      = float(df['close'].rolling(ma_period).mean().iloc[-1]) if len(df) >= ma_period else None
    daily_close = float(df['close'].iloc[-1])
    regime      = classify_trend(df['RSI'], lookback=trend_lookback)

    return sma200, daily_close, regime


def get_intraday_rsi(contract, rsi_period: int = RSI_PERIOD):
    """
    Fetch 5-min bars and compute RSI(14) for entry/exit signals.
    Returns (price_now, rsi_now, rsi_prev) or (None, None, None).
    """
    bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='5 D',           # enough warm-up bars for RSI(14)
        barSizeSetting='5 mins',
        whatToShow='TRADES',
        useRTH=False,
        formatDate=1
    )
    if not bars or len(bars) < rsi_period + 5:
        return None, None, None

    df         = util.df(bars)
    df.columns = [c.lower() for c in df.columns]
    df['RSI']  = calc_rsi(df['close'], period=rsi_period)
    df.dropna(subset=['RSI'], inplace=True)

    price    = float(df.iloc[-1]['close'])
    rsi_now  = float(df.iloc[-1]['RSI'])
    rsi_prev = float(df.iloc[-2]['RSI'])
    return price, rsi_now, rsi_prev


# ══════════════════════════════════════════════════════════════════════════════
# MONGO HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_doc(symbol, direction, entry_price, qty, rsi_at_entry, regime, sma200):
    ep = float(entry_price)
    return {
        'instrument':      symbol,
        'direction':       direction,
        'entry_price':     ep,
        'quantity':        int(qty),
        'remaining_qty':   int(qty),
        'rsi_at_entry':    float(rsi_at_entry),
        'regime_at_entry': regime,
        'sma200_at_entry': float(sma200) if sma200 else None,
        'entry_timestamp': datetime.now(),
        'order_id':        str(ObjectId()),
        # ── Live mark-to-market ──────────────────────────────────────────
        # Long:  peak_price   = highest close seen while trade is open
        # Short: trough_price = lowest  close seen while trade is open
        'current_price':       ep,
        'current_pnl':         0.0,
        'current_pnl_pct':     0.0,
        'peak_price':          ep,    # most favourable price (long=high, short=low)
        'trough_price':        ep,    # least favourable price (long=low, short=high)
        'max_unrealized_pnl':  0.0,   # best open profit ever seen ($)
        'min_unrealized_pnl':  0.0,   # deepest open loss ever seen ($, negative)
        'last_mark_timestamp': datetime.now(),
        # ────────────────────────────────────────────────────────────────
        'exit_price':      None,
        'exit_timestamp':  None,
        'exit_rsi':        None,
        'reason':          None,
        'realized_pnl':    None,
        'status':          'open',
        'created_at':      datetime.now(),
        'updated_at':      datetime.now(),
    }


def db_update(trade_id, updates: dict):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one({'_id': ObjectId(trade_id)}, {'$set': updates})


def update_trade_marks(trade_doc: dict, current_price: float) -> dict:
    """
    Recompute and persist all live mark-to-market fields for an open trade.

    Tracked fields
    ──────────────
    current_price       — latest bar close
    current_pnl         — open P&L in dollars
    current_pnl_pct     — open P&L as a fraction of entry cost
    peak_price          — most favourable price seen during the trade
                          (highest close for longs, lowest close for shorts)
    trough_price        — least favourable price seen during the trade
                          (lowest close for longs, highest close for shorts)
    max_unrealized_pnl  — largest open profit ever observed ($)
    min_unrealized_pnl  — largest open loss ever observed ($, stored negative)
    last_mark_timestamp — time of this update
    """
    entry_price = float(trade_doc['entry_price'])
    qty         = int(trade_doc['remaining_qty'])
    direction   = trade_doc.get('direction', 'long')
    cp          = float(current_price)

    # ── Direction-aware P&L ───────────────────────────────────────────────────
    if direction == 'long':
        pnl = (cp - entry_price) * qty
    else:  # short
        pnl = (entry_price - cp) * qty

    pnl_pct = pnl / (entry_price * qty) if entry_price * qty != 0 else 0.0

    # ── Peak / trough price ───────────────────────────────────────────────────
    prev_peak   = float(trade_doc.get('peak_price',   entry_price))
    prev_trough = float(trade_doc.get('trough_price', entry_price))

    if direction == 'long':
        new_peak   = max(prev_peak,   cp)   # long peak   = highest price  (best profit)
        new_trough = min(prev_trough, cp)   # long trough = lowest  price  (deepest loss)
    else:
        new_peak   = min(prev_peak,   cp)   # short peak  = lowest  price  (best profit)
        new_trough = max(prev_trough, cp)   # short trough= highest price  (deepest loss)

    # ── Max / min unrealized P&L ($) ─────────────────────────────────────────
    prev_max_pnl = float(trade_doc.get('max_unrealized_pnl', 0.0))
    prev_min_pnl = float(trade_doc.get('min_unrealized_pnl', 0.0))
    new_max_pnl  = max(prev_max_pnl, pnl)
    new_min_pnl  = min(prev_min_pnl, pnl)

    marks = {
        'current_price':       round(cp,       4),
        'current_pnl':         round(pnl,      2),
        'current_pnl_pct':     round(pnl_pct,  6),
        'peak_price':          round(new_peak,  4),
        'trough_price':        round(new_trough,4),
        'max_unrealized_pnl':  round(new_max_pnl, 2),
        'min_unrealized_pnl':  round(new_min_pnl, 2),
        'last_mark_timestamp': datetime.now(),
    }
    db_update(trade_doc['_id'], marks)
    return marks


def do_sell(contract, qty, entry_price, sell_price, rsi, reason, trade_id):
    """Close long position."""
    ib.placeOrder(contract, day_market_order('SELL', qty))
    pnl  = (sell_price - entry_price) * qty
    sign = '+' if pnl >= 0 else ''
    db_update(trade_id, {
        'exit_price':     float(sell_price),
        'exit_timestamp': datetime.now(),
        'exit_rsi':       float(rsi),
        'reason':         reason,
        'realized_pnl':   float(pnl),
        'status':         'closed',
    })
    print(f"  ✅ SELL (close long) {qty}sh @ ${sell_price:.2f}"
          f"  RSI={rsi:.1f}  [{reason}]  PnL: {sign}${pnl:.2f}")


def do_cover(contract, qty, entry_price, cover_price, rsi, reason, trade_id):
    """Cover short position."""
    ib.placeOrder(contract, day_market_order('BUY', qty))
    pnl  = (entry_price - cover_price) * qty
    sign = '+' if pnl >= 0 else ''
    db_update(trade_id, {
        'exit_price':     float(cover_price),
        'exit_timestamp': datetime.now(),
        'exit_rsi':       float(rsi),
        'reason':         reason,
        'realized_pnl':   float(pnl),
        'status':         'closed',
    })
    print(f"  ✅ COVER (close short) {qty}sh @ ${cover_price:.2f}"
          f"  RSI={rsi:.1f}  [{reason}]  PnL: {sign}${pnl:.2f}")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

async def check_and_trade():
    """
    ╔══════════════════════════════════════════════════════════════════════════╗
    ║  ANDREW CARDWELL — RSI RANGE SHIFT STRATEGY                             ║
    ╠══════════════════════════════════════════════════════════════════════════╣
    ║                                                                          ║
    ║  REGIME DETECTION (daily RSI over last 50 bars)                         ║
    ║    UPTREND   — RSI oscillates within  40–80  (bullish range)            ║
    ║    DOWNTREND — RSI oscillates within  20–60  (bearish range)            ║
    ║    NEUTRAL   — no clear range, sit out                                  ║
    ║                                                                          ║
    ║  ENTRY  (5-min RSI for precision timing)                                 ║
    ║    Long  — regime=UPTREND   + RSI pulls back into 40–50 zone            ║
    ║    Short — regime=DOWNTREND + RSI rallies into  50–60 zone              ║
    ║                                                                          ║
    ║  EXIT                                                                    ║
    ║    Long  — RSI reaches 70  (upper Cardwell uptrend bound)               ║
    ║    Short — RSI falls to 30 (lower Cardwell downtrend bound)             ║
    ║                                                                          ║
    ║  Position size: 40 shares fixed                                          ║
    ║  One trade per ticker per day                                            ║
    ║                                                                          ║
    ╚══════════════════════════════════════════════════════════════════════════╝
    """

    while True:
        ib.positions()   # refresh internal positions cache

        for contract in contracts:
            symbol = contract.symbol
            print(f"\n{'='*60}")
            print(f"  {symbol}  {datetime.now().strftime('%H:%M:%S')}")
            print(f"{'='*60}")

            # ── 1. Daily regime detection ─────────────────────────────────
            sma200, daily_close, regime = get_daily_data(contract)
            if regime is None:
                print(f"  Insufficient daily data — skipping {symbol}")
                continue

            sma_str = f"SMA200=${sma200:.2f}  " if sma200 else ""
            print(f"  {sma_str}DailyClose=${daily_close:.2f}  Regime={regime}")

            if regime == 'NEUTRAL':
                print(f"  RSI range is NEUTRAL — no clear trend, skipping.")
                continue

            # ── 2. Intraday RSI(14) for entry timing ──────────────────────
            price, rsi_now, rsi_prev = get_intraday_rsi(contract)
            if price is None:
                print(f"  Insufficient intraday data — skipping {symbol}")
                continue

            print(f"  Price=${price:.2f}  RSI={rsi_now:.1f}  RSI_prev={rsi_prev:.1f}")

            open_trade = trades_collection.find_one({'instrument': symbol, 'status': 'open'})

            # ══════════════════════════════════════════════════════════════
            # POSITION MANAGEMENT
            # ══════════════════════════════════════════════════════════════

            if open_trade:
                entry_price = float(open_trade['entry_price'])
                qty         = int(open_trade['remaining_qty'])
                direction   = open_trade.get('direction', 'long')
                trade_id    = open_trade['_id']

                if direction == 'long':
                    # ── Update live marks every scan ────────────────────
                    marks = update_trade_marks(open_trade, price)

                    print(f"  OPEN LONG  entry=${entry_price:.2f}  qty={qty}")
                    print(f"    Current : ${marks['current_price']:.2f}  "
                          f"P&L={marks['current_pnl_pct']:+.2%}  (${marks['current_pnl']:+.2f})")
                    print(f"    Peak    : ${marks['peak_price']:.2f}  "
                          f"MaxProfit=${marks['max_unrealized_pnl']:+.2f}")
                    print(f"    Trough  : ${marks['trough_price']:.2f}  "
                          f"MaxLoss  =${marks['min_unrealized_pnl']:+.2f}")

                    # Exit long: RSI reaches the upper Cardwell uptrend boundary
                    if rsi_now >= LONG_EXIT_RSI:
                        do_sell(contract, qty, entry_price, price,
                                rsi_now, f'RSI_REACHED_{LONG_EXIT_RSI}_LONG_EXIT', trade_id)
                        print(f"  📈 LONG EXIT: RSI={rsi_now:.1f} ≥ {LONG_EXIT_RSI}")

                    # Safety: regime flipped to downtrend — exit stale long
                    elif regime == 'DOWNTREND':
                        do_sell(contract, qty, entry_price, price,
                                rsi_now, 'REGIME_FLIP_TO_DOWNTREND', trade_id)
                        print(f"  ⚠️  LONG EXIT: regime flipped to DOWNTREND")

                    else:
                        print(f"  Holding long — waiting RSI ≥ {LONG_EXIT_RSI}  (RSI={rsi_now:.1f})")

                elif direction == 'short':
                    # ── Update live marks every scan ────────────────────
                    marks = update_trade_marks(open_trade, price)

                    print(f"  OPEN SHORT  entry=${entry_price:.2f}  qty={qty}")
                    print(f"    Current : ${marks['current_price']:.2f}  "
                          f"P&L={marks['current_pnl_pct']:+.2%}  (${marks['current_pnl']:+.2f})")
                    print(f"    Peak    : ${marks['peak_price']:.2f}  "
                          f"MaxProfit=${marks['max_unrealized_pnl']:+.2f}")
                    print(f"    Trough  : ${marks['trough_price']:.2f}  "
                          f"MaxLoss  =${marks['min_unrealized_pnl']:+.2f}")

                    # Exit short: RSI falls to the lower Cardwell downtrend boundary
                    if rsi_now <= SHORT_EXIT_RSI:
                        do_cover(contract, qty, entry_price, price,
                                 rsi_now, f'RSI_FELL_TO_{SHORT_EXIT_RSI}_SHORT_EXIT', trade_id)
                        print(f"  📉 SHORT EXIT: RSI={rsi_now:.1f} ≤ {SHORT_EXIT_RSI}")

                    # Safety: regime flipped to uptrend — cover stale short
                    elif regime == 'UPTREND':
                        do_cover(contract, qty, entry_price, price,
                                 rsi_now, 'REGIME_FLIP_TO_UPTREND', trade_id)
                        print(f"  ⚠️  SHORT EXIT: regime flipped to UPTREND")

                    else:
                        print(f"  Holding short — waiting RSI ≤ {SHORT_EXIT_RSI}  (RSI={rsi_now:.1f})")

            # ══════════════════════════════════════════════════════════════
            # ENTRY LOGIC
            # ══════════════════════════════════════════════════════════════

            else:
                today = datetime.now().date()
                if exclude_collection.find_one({'ticker': symbol, 'date': today.isoformat()}):
                    print(f"  Already traded {symbol} today — skipping.")
                    continue

                in_long_zone  = LONG_ENTRY_LOW  <= rsi_now <= LONG_ENTRY_HIGH
                in_short_zone = SHORT_ENTRY_LOW <= rsi_now <= SHORT_ENTRY_HIGH

                # ── LONG: uptrend + RSI pulls back to 40–50 support ───────
                if regime == 'UPTREND' and in_long_zone:
                    ib.placeOrder(contract, day_market_order('BUY', ORDER_SIZE))

                    doc = create_trade_doc(symbol, 'long', price, ORDER_SIZE,
                                           rsi_now, regime, sma200)
                    trades_collection.insert_one(sanitize_for_mongo(doc))
                    exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                    print(f"  🚀 BUY (long)  {symbol}")
                    print(f"     Entry:  ${price:.2f}  |  Regime=UPTREND")
                    print(f"     Shares: {ORDER_SIZE}")
                    print(f"     RSI:    {rsi_now:.1f}  (pullback to 40–50 support zone)")
                    print(f"     Target: RSI ≥ {LONG_EXIT_RSI}")

                # ── SHORT: downtrend + RSI rallies to 50–60 resistance ────
                elif regime == 'DOWNTREND' and in_short_zone:
                    ib.placeOrder(contract, day_market_order('SELL', ORDER_SIZE))

                    doc = create_trade_doc(symbol, 'short', price, ORDER_SIZE,
                                           rsi_now, regime, sma200)
                    trades_collection.insert_one(sanitize_for_mongo(doc))
                    exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                    print(f"  🔻 SELL (short)  {symbol}")
                    print(f"     Entry:  ${price:.2f}  |  Regime=DOWNTREND")
                    print(f"     Shares: {ORDER_SIZE}")
                    print(f"     RSI:    {rsi_now:.1f}  (rally to 50–60 resistance zone)")
                    print(f"     Target: RSI ≤ {SHORT_EXIT_RSI}")

                else:
                    if regime == 'UPTREND':
                        print(f"  No entry — UPTREND but RSI={rsi_now:.1f}"
                              f"  (waiting for 40–50 pullback zone)")
                    else:
                        print(f"  No entry — DOWNTREND but RSI={rsi_now:.1f}"
                              f"  (waiting for 50–60 rally zone)")

            await asyncio.sleep(0.5)   # brief pause between tickers

        print(f"\n{'='*60}")
        print(f"  Scan complete. Next scan in 5 minutes.")
        print(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════

try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()
finally:
    ib.disconnect()
    print("Disconnected from IB.")


Error 200, reqId 5: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Error 200, reqId 31: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')



  WMT  07:20:36
  SMA200=$108.54  DailyClose=$122.18  Regime=UPTREND
  Price=$122.00  RSI=30.8  RSI_prev=30.8
  No entry — UPTREND but RSI=30.8  (waiting for 40–50 pullback zone)

  MU  07:20:37
  SMA200=$237.25  DailyClose=$355.46  Regime=UPTREND
  Price=$347.40  RSI=30.3  RSI_prev=30.3
  OPEN LONG  entry=$356.52  qty=40
    Current : $347.40  P&L=-2.56%  ($-364.80)
    Peak    : $356.52  MaxProfit=$+0.00
    Trough  : $347.40  MaxLoss  =$-364.80
  Holding long — waiting RSI ≥ 70  (RSI=30.3)


Error 200, reqId 78: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')



  SAVA  07:20:38
  Insufficient daily data — skipping SAVA

  SLDB  07:20:38
  SMA200=$5.79  DailyClose=$7.73  Regime=UPTREND
  Price=$7.73  RSI=59.5  RSI_prev=59.5
  No entry — UPTREND but RSI=59.5  (waiting for 40–50 pullback zone)

  SLV  07:20:39
  SMA200=$52.02  DailyClose=$60.77  Regime=DOWNTREND
  Price=$61.62  RSI=32.2  RSI_prev=31.7
  OPEN SHORT  entry=$62.14  qty=40
    Current : $61.62  P&L=+0.84%  ($+20.80)
    Peak    : $61.62  MaxProfit=$+20.80
    Trough  : $62.14  MaxLoss  =$+0.00
  Holding short — waiting RSI ≤ 30  (RSI=32.2)

  NEM  07:20:40
  SMA200=$88.83  DailyClose=$99.36  Regime=DOWNTREND
  Price=$99.34  RSI=39.5  RSI_prev=39.5
  No entry — DOWNTREND but RSI=39.5  (waiting for 50–60 rally zone)

  CTXR  07:20:41
  SMA200=$1.16  DailyClose=$0.69  Regime=DOWNTREND
  Price=$0.70  RSI=30.4  RSI_prev=30.4
  No entry — DOWNTREND but RSI=30.4  (waiting for 50–60 rally zone)

  ONON  07:20:42
  SMA200=$45.56  DailyClose=$32.11  Regime=DOWNTREND
  Price=$31.95  RSI=24.8 

Error 200, reqId 134: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')


...

  Price=$91.17  RSI=28.5  RSI_prev=28.5
  Already traded UAL today — skipping.

  BRK.B  08:58:01
  Insufficient daily data — skipping BRK.B

  IBM  08:58:01
  SMA200=$278.52  DailyClose=$241.67  Regime=DOWNTREND
  Price=$237.51  RSI=35.7  RSI_prev=35.7
  No entry — DOWNTREND but RSI=35.7  (waiting for 50–60 rally zone)

  MSFT  08:58:02
  SMA200=$479.65  DailyClose=$365.97  Regime=DOWNTREND
  Price=$361.85  RSI=25.2  RSI_prev=23.9
  No entry — DOWNTREND but RSI=25.2  (waiting for 50–60 rally zone)

  KO  08:58:03
  SMA200=$71.18  DailyClose=$74.69  Regime=UPTREND
  Price=$74.95  RSI=67.6  RSI_prev=67.6
  OPEN LONG  entry=$74.78  qty=40
    Current : $74.95  P&L=+0.23%  ($+6.80)
    Peak    : $74.95  MaxProfit=$+6.80
    Trough  : $74.77  MaxLoss  =$-0.40
  Holding long — waiting RSI ≥ 70  (RSI=67.6)

  MSTR  08:58:04
  SMA200=$260.86  DailyClose=$132.93  Regime=DOWNTREND
  Price=$129.10  RSI=44.6  RSI_prev=46.3
  No entry — DOWNTREND but RSI=44.6  (waiting for 50–60 rally zone)

  COI

Error 200, reqId 2471: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')



  SAVA  09:03:40
  Insufficient daily data — skipping SAVA

  SLDB  09:03:40
  SMA200=$5.79  DailyClose=$7.73  Regime=UPTREND
  Price=$7.73  RSI=59.5  RSI_prev=59.5
  No entry — UPTREND but RSI=59.5  (waiting for 40–50 pullback zone)

  SLV  09:03:40
  SMA200=$52.02  DailyClose=$60.77  Regime=DOWNTREND
  Price=$61.51  RSI=44.6  RSI_prev=42.6
  OPEN SHORT  entry=$62.14  qty=40
    Current : $61.51  P&L=+1.01%  ($+25.20)
    Peak    : $61.30  MaxProfit=$+33.60
    Trough  : $62.14  MaxLoss  =$+0.00
  Holding short — waiting RSI ≤ 30  (RSI=44.6)

  NEM  09:03:41
  SMA200=$88.83  DailyClose=$99.36  Regime=DOWNTREND
  Price=$99.69  RSI=49.8  RSI_prev=49.9
  OPEN SHORT  entry=$99.72  qty=40
    Current : $99.69  P&L=+0.03%  ($+1.20)
    Peak    : $99.21  MaxProfit=$+20.40
    Trough  : $99.94  MaxLoss  =$-8.80
  Holding short — waiting RSI ≤ 30  (RSI=49.8)

  CTXR  09:03:42
  SMA200=$1.16  DailyClose=$0.69  Regime=DOWNTREND
  Price=$0.70  RSI=38.0  RSI_prev=38.0
  OPEN SHORT  entry=$0.71  q

Error 200, reqId 2522: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')


  Price=$91.58  RSI=46.4  RSI_prev=46.4
  Already traded UAL today — skipping.

  BRK.B  09:04:01
  Insufficient daily data — skipping BRK.B

  IBM  09:04:01
  SMA200=$278.52  DailyClose=$241.67  Regime=DOWNTREND
  Price=$237.70  RSI=37.8  RSI_prev=35.7
  No entry — DOWNTREND but RSI=37.8  (waiting for 50–60 rally zone)

  MSFT  09:04:02
  SMA200=$479.65  DailyClose=$365.97  Regime=DOWNTREND
  Price=$361.83  RSI=25.8  RSI_prev=26.5
  No entry — DOWNTREND but RSI=25.8  (waiting for 50–60 rally zone)

  KO  09:04:03
  SMA200=$71.18  DailyClose=$74.69  Regime=UPTREND
  Price=$74.98  RSI=70.3  RSI_prev=67.6
  OPEN LONG  entry=$74.78  qty=40
    Current : $74.98  P&L=+0.27%  ($+8.00)
    Peak    : $74.98  MaxProfit=$+8.00
    Trough  : $74.77  MaxLoss  =$-0.40
  ✅ SELL (close long) 40sh @ $74.98  RSI=70.3  [RSI_REACHED_70_LONG_EXIT]  PnL: +$8.00
  📈 LONG EXIT: RSI=70.3 ≥ 70

  MSTR  09:04:04
  SMA200=$260.86  DailyClose=$132.93  Regime=DOWNTREND
